# S3.23 - Notebook exit values
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** A child notebook reports back to its parent with dbutils.notebook.exit(string)

In [0]:
# ============================================================
# S3.23 — Parent / Checkpoint Cell
#
# Goal:
# 1. Create S3.23_child_notebook next to this notebook.
# 2. Run the child with mode="ok".
# 3. Verify the returned value is a STRING.
# 4. Decode the returned JSON.
# 5. Run the child with mode="fail".
# 6. Verify the child failure reaches the parent as an exception.
#
# Important:
#
# dbutils.notebook.run(...)
# → returns the child's exit value as a STRING.
#
# dbutils.notebook.exit(...)
# → accepts one STRING.
#
# JSON allows several logical result fields to travel inside
# that one returned string.
# ============================================================

import base64
import json

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import (
    ImportFormat,
    Language
)

from dbruntime.databricks_repl_context import (
    get_context
)


print("=== S3.23: PARENT / CHILD CHECKPOINT ===")
print()


# ------------------------------------------------------------
# Step 1 — Child Notebook Source
#
# mode="ok"
# → count orders
# → return JSON string
#
# mode="fail"
# → raise ValueError
# → parent run() should receive an exception
# ------------------------------------------------------------

child_src = '''
import json

dbutils.widgets.text("mode", "ok")

mode = (
    dbutils.widgets
    .get("mode")
    .strip()
    .lower()
)


if mode not in {"ok", "fail"}:

    raise ValueError(
        f"Invalid mode: {mode}"
    )


if mode == "fail":

    raise ValueError(
        "simulated failure"
    )


row_count = (
    spark.table(
        "samples.tpch.orders"
    )
    .count()
)


result = {
    "status": "SUCCESS",
    "rows": row_count
}


dbutils.notebook.exit(
    json.dumps(result)
)
'''


# ------------------------------------------------------------
# Step 2 — Find Current Notebook Folder
#
# Use the Databricks REPL context rather than the older
# dbutils.notebook.entry_point pattern.
# ------------------------------------------------------------

context = (
    get_context()
)


current_notebook_path = (
    context.notebookPath
)


if not current_notebook_path:

    raise RuntimeError(
        "Current notebook path could not be determined."
    )


parent_folder = (
    current_notebook_path
    .rsplit("/", 1)[0]
)


child_path = (
    f"{parent_folder}/S3.23_child_notebook"
)


print(
    f"Current notebook : "
    f"{current_notebook_path}"
)

print(
    f"Child notebook   : "
    f"{child_path}"
)

print()


# ------------------------------------------------------------
# Step 3 — Create / Overwrite Child Notebook
#
# Databricks Workspace API SOURCE imports expect the content
# to be base64 encoded.
# ------------------------------------------------------------

workspace = (
    WorkspaceClient()
)


encoded_child_source = (
    base64
    .b64encode(
        child_src.encode("utf-8")
    )
    .decode("utf-8")
)


print(
    "=== STEP 3: CREATE CHILD NOTEBOOK ==="
)


try:

    workspace.workspace.import_(
        path=child_path,
        content=encoded_child_source,
        format=ImportFormat.SOURCE,
        language=Language.PYTHON,
        overwrite=True
    )

    child_created = True

    print(
        "Child notebook creation: SUCCESS"
    )


except Exception as e:

    child_created = False

    print(
        "Child notebook creation: FAILED"
    )

    print(
        f"Exception type: "
        f"{type(e).__name__}"
    )

    print(
        f"Message preview: "
        f"{str(e)[:300]}"
    )

    raise


print()


# ------------------------------------------------------------
# Step 4 — SUCCESS Checkpoint
#
# Parent passes:
#
# mode = ok
#
# Child should:
#
# count orders
# → exit(JSON string)
#
# Parent should receive:
#
# str
# ------------------------------------------------------------

print(
    "=== STEP 4: SUCCESS PATH — mode=ok ==="
)


ok_verified = False
ok_raw = None
ok_result = None


try:

    ok_raw = (
        dbutils.notebook.run(
            child_path,
            300,
            {
                "mode": "ok"
            }
        )
    )


    print(
        f"Returned Python type: "
        f"{type(ok_raw).__name__}"
    )

    print(
        f"Raw returned value : "
        f"{ok_raw}"
    )


    ok_result = (
        json.loads(
            ok_raw
        )
    )


    print(
        f"Parsed JSON        : "
        f"{ok_result}"
    )

    print(
        f"Decoded status     : "
        f"{ok_result.get('status')}"
    )

    print(
        f"Decoded rows       : "
        f"{ok_result.get('rows')}"
    )


    ok_verified = (
        isinstance(
            ok_raw,
            str
        )
        and
        ok_result.get("status")
        ==
        "SUCCESS"
        and
        isinstance(
            ok_result.get("rows"),
            int
        )
    )


except Exception as e:

    print(
        "Unexpected failure on mode=ok"
    )

    print(
        f"Exception type: "
        f"{type(e).__name__}"
    )

    print(
        f"Message preview: "
        f"{str(e)[:300]}"
    )


print(
    f"SUCCESS checkpoint verified: "
    f"{ok_verified}"
)

print()


# ------------------------------------------------------------
# Step 5 — FAILURE Checkpoint
#
# Parent passes:
#
# mode = fail
#
# Child should:
#
# raise ValueError
#
# Therefore dbutils.notebook.run(...)
# should raise in the parent.
# ------------------------------------------------------------

print(
    "=== STEP 5: FAILURE PATH — mode=fail ==="
)


fail_verified = False
fail_exception_type = None
fail_exception_message = None


try:

    unexpected_result = (
        dbutils.notebook.run(
            child_path,
            300,
            {
                "mode": "fail"
            }
        )
    )


    print(
        "UNEXPECTED: failure-mode child returned normally."
    )

    print(
        f"Returned value: "
        f"{unexpected_result}"
    )


except Exception as e:

    fail_verified = True

    fail_exception_type = (
        type(e).__name__
    )

    fail_exception_message = (
        str(e).splitlines()[0][:300]
    )


    print(
        "Expected child failure reached the parent."
    )

    print(
        f"Parent exception type: "
        f"{fail_exception_type}"
    )

    print(
        f"Message preview: "
        f"{fail_exception_message}"
    )


print(
    f"FAILURE checkpoint verified: "
    f"{fail_verified}"
)

print()


# ------------------------------------------------------------
# Step 6 — Final Checkpoint
# ------------------------------------------------------------

print(
    "=== STEP 6: S3.23 CHECKPOINT ==="
)


print(
    f"Child created            : "
    f"{child_created}"
)

print(
    f"Success path verified    : "
    f"{ok_verified}"
)

print(
    f"Failure path verified    : "
    f"{fail_verified}"
)


if (
    child_created
    and
    ok_verified
    and
    fail_verified
):

    print("\nS3.23 checkpoint PASSED.")

else:

    print("\nS3.23 checkpoint NOT YET PASSED.")


print("=== S3.23 PARENT / CHILD TEST COMPLETED ===")

# S3.23 — Key Takeaways: Notebook Exit Values

## 1. exit() and run()

The child notebook can return a value using:

```python
dbutils.notebook.exit(value)
```

The parent receives that value from:

```python
dbutils.notebook.run(...)
```

`dbutils.notebook.exit()` returns one string value.

Therefore, when several logical values need to be returned, encode them as JSON:

```python
result = {
    "status": "SUCCESS",
    "rows": row_count
}

dbutils.notebook.exit(
    json.dumps(result)
)
```

The parent can decode the returned JSON using:

```python
json.loads(...)
```

---

## 2. Parameters

The parent passes parameters through `dbutils.notebook.run()`.

Example:

```python
dbutils.notebook.run(
    child_path,
    300,
    {
        "mode": "ok"
    }
)
```

The child receives the parameter through a widget:

```python
dbutils.widgets.text(
    "mode",
    "ok"
)

mode = dbutils.widgets.get(
    "mode"
)
```

Conceptually:

```text
Parent arguments
      ↓
Child widgets
      ↓
Child logic
```

---

## 3. Success Path

Our child logic for:

```text
mode = ok
```

does:

```text
read samples.tpch.orders
      ↓
count rows
      ↓
build JSON result
      ↓
dbutils.notebook.exit(...)
```

Direct child execution produced:

```json
{"status": "SUCCESS", "rows": 7500000}
```

The Notebook Workflows view also showed:

```text
mode = ok
Status = Succeeded
```

Therefore the child success path is verified.

---

## 4. Failure Path

Our controlled failure uses:

```python
if mode == "fail":
    raise ValueError(
        "simulated failure"
    )
```

A real execution failure should be raised rather than disguised as a successful notebook exit.

Conceptually:

```text
mode = fail
      ↓
raise ValueError
      ↓
child execution fails
      ↓
parent notebook run fails
```

The Notebook Workflows view showed:

```text
mode = fail
Status = Failed
Error code = RunExecutionError
```

Therefore the failure path is verified at the notebook-workflow level.

---

## 5. exit() vs raise

These represent different outcomes.

### exit()

```text
Notebook completed normally
      ↓
return control/result information
```

Example:

```json
{"status": "SUCCESS", "rows": 7500000}
```

### raise

```text
Notebook execution failed
      ↓
failure propagates to the caller/workflow
```

Easy way to remember:

```text
exit()
→ normal communication

raise
→ execution failure
```

---

## 6. Do Not Return Large Data Through exit()

Use `notebook.exit()` for small control information such as:

```text
status
row count
table name
path
run ID
small JSON metadata
```

Do not use it to return a large DataFrame.

For large results:

```text
persist data
      ↓
return table/path/reference
```

---

## 7. Observed in Our Workspace

Verified:

```text
Direct child execution
→ SUCCESS

Returned child JSON
→ {"status": "SUCCESS", "rows": 7500000}

mode = ok workflow run
→ Succeeded

mode = fail workflow run
→ Failed

Failure error code
→ RunExecutionError

Run parameter mode
→ passed correctly to the child
```

Earlier during development we also encountered a notebook-path/access issue when the child was outside the accessible repo location.

That was a setup/path issue, not the final `run()` / `exit()` behavior.

---

## 8. Still Not Verified

From the evidence currently captured, we have not yet recorded the exact parent-side success return details:

```text
type returned by dbutils.notebook.run()
= ______

raw returned value
= ______

json.loads(...) result
= ______
```

These are useful to verify when we revisit notebook orchestration in the Jobs and Workflows section.

---

## 9. Checkpoint

### Q1. How does a child notebook return information to a parent?

Using:

```python
dbutils.notebook.exit(...)
```

The parent receives the value from:

```python
dbutils.notebook.run(...)
```

---

### Q2. Why use `json.dumps()`?

Because the notebook exit contract returns one string.

JSON allows multiple logical fields such as:

```text
status
rows
message
```

to be encoded inside that one string.

---

### Q3. How are parameters passed to the child?

The parent supplies arguments to `dbutils.notebook.run()`.

The child reads them through widgets.

---

### Q4. What should happen when the child has a real execution failure?

Raise an exception.

Do not turn an actual execution failure into a fake successful return value.

---

### Q5. What happened in our `mode="fail"` test?

The child run was marked:

```text
Failed
```

with:

```text
RunExecutionError
```

So the failure propagated through the notebook workflow.

---

### Q6. What happened in our `mode="ok"` test?

The workflow run was marked:

```text
Succeeded
```

and direct child execution returned:

```json
{"status": "SUCCESS", "rows": 7500000}
```

---

## 10. Architecture Note

For this section, the main concept was:

```text
parent
→ child parameter
→ child execution
→ exit value or failure
```

More advanced orchestration topics such as:

```text
task dependencies
retries
scheduling
job-level monitoring
workflow design
```

can be covered in the Jobs and Workflows section.

---

## 11. Core Understanding

```text
Need to pass input to child?
→ arguments + widgets

Need normal child result?
→ dbutils.notebook.exit()

Need multiple logical result values?
→ json.dumps()

Need actual execution failure?
→ raise exception

Need parent/child orchestration?
→ dbutils.notebook.run()

Need production workflow orchestration?
→ Jobs and Workflows
```

## Final Checkpoint Status

Verified:

```text
Child direct success
→ Yes

mode=ok workflow success
→ Yes

mode=fail workflow failure
→ Yes

Failure surfaced as RunExecutionError
→ Yes
```

Remaining parent return-value inspection can be revisited with the Jobs and Workflows practical.

**S3.23 — Notebook Exit Values checkpoint completed for the current scope.**